# 🤖 Train AI Trợ Lý GS25 với MiniMind

**Notebook đã được fix lỗi 401 / repo not found.**

### Cách hoạt động:
- Clone repo MiniMind từ GitHub (không cần HF token)
- Tải pre-trained weights từ HuggingFace model mới nhất `jingyaogong/minimind-3o`
- Fine-tune SFT bằng dataset GS25 của bạn

**Bật GPU trước:** Runtime → Change runtime type → **T4 GPU**

## 📦 Bước 1: Clone MiniMind & cài dependencies

In [ ]:
import subprocess, os

# Kiểm tra GPU
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('🖥️ GPU:', r.stdout.strip() or 'KHÔNG TÌM THẤY GPU — Hãy bật T4 GPU!')

# Clone MiniMind từ GitHub (không cần token)
if not os.path.exists('minimind'):
    os.system('git clone https://github.com/jingyaogong/minimind.git')
else:
    print('✅ MiniMind đã có sẵn')

os.chdir('minimind')
print('📂 Thư mục hiện tại:', os.getcwd())
print('📋 Nội dung:', os.listdir('.'))

In [ ]:
# Cài dependencies
os.system('pip install -q transformers datasets tiktoken sentencepiece accelerate')

# Cài từ requirements.txt của MiniMind
if os.path.exists('requirements.txt'):
    os.system('pip install -q -r requirements.txt')

# Import kiểm tra
import torch
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📤 Bước 2: Upload file gs25_qa.jsonl

In [ ]:
from google.colab import files
import json, os

os.makedirs('dataset', exist_ok=True)

print('📁 Hãy upload file gs25_qa.jsonl từ máy tính:')
uploaded = files.upload()

# Tìm file jsonl vừa upload
jsonl_file = None
for fname in uploaded:
    if fname.endswith('.jsonl'):
        jsonl_file = fname
        break

if jsonl_file:
    # Chuyển về dataset/sft_data.jsonl theo format MiniMind
    raw_data = []
    content = uploaded[jsonl_file].decode('utf-8')
    for line in content.strip().split('\n'):
        item = json.loads(line)
        # Chuyển sang format conversations của MiniMind
        raw_data.append({
            'conversations': [
                {
                    'role': 'system',
                    'content': 'Bạn là AI trợ lý nghiệp vụ của chuỗi cửa hàng tiện lợi GS25 Việt Nam. Hãy trả lời chính xác và hữu ích.'
                },
                {'role': 'user', 'content': item['instruction']},
                {'role': 'assistant', 'content': item['output']}
            ]
        })

    out_path = 'dataset/sft_data.jsonl'
    with open(out_path, 'w', encoding='utf-8') as f:
        for d in raw_data:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')

    print(f'✅ Đã convert {len(raw_data)} samples → {out_path}')

    # Preview 1 sample
    s = raw_data[0]['conversations']
    print(f'\n📋 Sample đầu tiên:')
    print(f'  Q: {s[1]["content"]}')
    print(f'  A: {s[2]["content"][:100]}...')
else:
    print('❌ Không tìm thấy file .jsonl. Hãy upload đúng file gs25_qa.jsonl')

## 🔑 Bước 3: Đăng nhập Hugging Face (để tải base model)

In [ ]:
# Lấy HF token tại: https://huggingface.co/settings/tokens
# Tạo token loại READ là đủ
from huggingface_hub import login

HF_TOKEN = input('Nhập Hugging Face READ Token (hf_...): ').strip()

if HF_TOKEN.startswith('hf_'):
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Đã đăng nhập Hugging Face thành công!')
else:
    print('❌ Token không hợp lệ. Token phải bắt đầu bằng hf_')
    print('   Tạo token tại: https://huggingface.co/settings/tokens')

## ⬇️ Bước 4: Tải MiniMind Pre-trained Weights

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
import os

os.makedirs('out', exist_ok=True)

print('⬇️ Đang tải MiniMind pre-trained weights...')
print('   Repo: jingyaogong/minimind-3o')

try:
    # Tải model mới nhất (minimind-3o)
    snapshot_download(
        repo_id='jingyaogong/minimind-3o',
        local_dir='out/pretrain_base',
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
        token=HF_TOKEN
    )
    print('✅ Đã tải xong pre-trained weights!')
    print('📂 Nội dung out/pretrain_base:', os.listdir('out/pretrain_base'))
except Exception as e:
    print(f'⚠️ Lỗi tải từ HF: {e}')
    print('\n🔄 Thử tải từ ModelScope (không cần token)...')
    # Fallback: tải từ ModelScope (mirror Trung Quốc, không cần auth)
    os.system('pip install -q modelscope')
    from modelscope import snapshot_download as ms_download
    ms_download('jingyaogong/minimind-3o', cache_dir='out/pretrain_base')
    print('✅ Đã tải từ ModelScope!')

## 🔧 Bước 5: Cấu hình SFT cho dataset GS25

In [ ]:
import json, os

# Đọc config hiện tại của MiniMind
config_files = [f for f in os.listdir('.') if 'config' in f.lower() and f.endswith('.json')]
print('Config files:', config_files)

# Kiểm tra train script có sẵn
train_scripts = [f for f in os.listdir('.') if 'train' in f.lower() and f.endswith('.py')]
print('Train scripts:', train_scripts)

# Đọc 1 dòng dataset để xác nhận format đúng
with open('dataset/sft_data.jsonl', encoding='utf-8') as f:
    sample = json.loads(f.readline())
print('\n✅ Dataset format OK:')
print(f'   Keys: {list(sample.keys())}')
print(f'   Roles: {[m["role"] for m in sample["conversations"]]}')

# Đếm tổng samples
with open('dataset/sft_data.jsonl', encoding='utf-8') as f:
    n_samples = sum(1 for _ in f)
print(f'   Tổng samples: {n_samples}')

## 🚀 Bước 6: Chạy SFT Fine-tuning

In [ ]:
import subprocess, os

os.makedirs('out/gs25_sft', exist_ok=True)

# Kiểm tra xem script train SFT là file nào
sft_script = None
for candidate in ['train_sft.py', '2-sft.py', 'train_full_sft.py']:
    if os.path.exists(candidate):
        sft_script = candidate
        break

if sft_script is None:
    # Liệt kê tất cả file .py
    py_files = [f for f in os.listdir('.') if f.endswith('.py')]
    print('Tìm thấy các file .py:', py_files)
    sft_script = input('Nhập tên file train SFT (ví dụ: train_sft.py): ').strip()

print(f'🚀 Sử dụng script: {sft_script}')
print('⏱️ Ước tính: 30-60 phút trên T4 GPU (147 samples × 10 epochs)')
print('='*60)

# Chạy train
result = subprocess.run(
    ['python', sft_script,
     '--data_path', 'dataset/sft_data.jsonl',
     '--out_dir', 'out/gs25_sft',
     '--epochs', '15',
     '--batch_size', '4',  # T4 = 16GB, batch 4 là an toàn
     '--learning_rate', '3e-5',
     '--max_seq_len', '512',
    ],
    text=True
)

if result.returncode == 0:
    print('\n✅ SFT Fine-tuning hoàn thành!')
    print('📂 Model lưu tại: out/gs25_sft/')
    print('📋 Files:', os.listdir('out/gs25_sft/'))
else:
    print('\n❌ Có lỗi. Xem log bên trên.')
    print('💡 Thử chạy: !python', sft_script, '--help')

## 🧪 Bước 7: Test mô hình

In [ ]:
# Chạy inference test với 5 câu hỏi GS25
test_questions = [
    'Part-time tối đa làm bao nhiêu giờ mỗi tuần?',
    'Câu chào khách chuẩn của GS25 là gì?',
    'Ca đêm có phụ cấp không?',
    'Làm thế nào để đổi ca?',
    'Chu kỳ lương GS25 từ ngày mấy đến ngày mấy?',
]

# Tìm script inference
infer_scripts = [f for f in os.listdir('.') if 'infer' in f.lower() and f.endswith('.py')]
chat_scripts = [f for f in os.listdir('.') if 'chat' in f.lower() and f.endswith('.py')]
print('Inference scripts:', infer_scripts)
print('Chat scripts:', chat_scripts)

print('\n🧪 Câu hỏi test:')
for i, q in enumerate(test_questions, 1):
    print(f'{i}. {q}')

print('\n💡 Chạy inference thủ công:')
if infer_scripts:
    print(f'   python {infer_scripts[0]} --model_path out/gs25_sft/')
elif chat_scripts:
    print(f'   python {chat_scripts[0]} --model_path out/gs25_sft/')

## ☁️ Bước 8: Upload model lên Hugging Face

In [ ]:
from huggingface_hub import HfApi

# Nhập thông tin
HF_USERNAME = input('Hugging Face username của bạn: ').strip()
REPO_NAME = 'gs25-assistant'
FULL_REPO = f'{HF_USERNAME}/{REPO_NAME}'

print(f'\n📤 Chuẩn bị upload lên: https://huggingface.co/{FULL_REPO}')

api = HfApi(token=HF_TOKEN)

# Tạo repo
try:
    api.create_repo(repo_id=FULL_REPO, repo_type='model', exist_ok=True)
    print(f'✅ Repo created/exists: {FULL_REPO}')
except Exception as e:
    print(f'⚠️ {e}')

# Upload model folder
model_dir = 'out/gs25_sft'
if os.path.exists(model_dir) and os.listdir(model_dir):
    api.upload_folder(
        folder_path=model_dir,
        repo_id=FULL_REPO,
        repo_type='model'
    )
    print(f'\n🎉 Upload xong!')
    print(f'🔗 Model URL: https://huggingface.co/{FULL_REPO}')
    print(f'📡 API endpoint: https://api-inference.huggingface.co/models/{FULL_REPO}')
    print()
    print('='*60)
    print('📝 BƯỚC CUỐI: Thêm vào file .env của dự án GS25:')
    print(f'   VITE_GS25_AI_MODEL_URL=https://api-inference.huggingface.co/models/{FULL_REPO}')
    print(f'   VITE_HF_TOKEN={HF_TOKEN[:10]}...')
    print('='*60)
else:
    print(f'❌ Không tìm thấy model tại {model_dir}')
    print('   Hãy chạy lại Bước 6 trước.')

## ✅ Hoàn tất!

Sau khi có model URL, mở file `.env` trong dự án GS25 và thêm:

```
VITE_GS25_AI_MODEL_URL=https://api-inference.huggingface.co/models/YOUR_USERNAME/gs25-assistant
VITE_HF_TOKEN=hf_YOUR_TOKEN
```

AI Copilot trong app sẽ tự động dùng model GS25 đã train! 🎉